# V10: 불필요 날씨 피처 제거 및 앙상블(XGBoost + CatBoost) 모델링

이 노트북은 V9의 결과를 분석하여, 기여도가 낮은 날씨 변수를 제거하고 '연휴 전후' 피처를 추가하여 점수를 한 단계 더 높이는 것을 목표로 합니다.

## 주요 전략
1. **날씨 피처 정제**: 기여도가 낮은 `습도`, `불쾌지수`, `연속형 강수량` 제거. 핵심인 `기온`과 `비온날(Binary)`만 유지
2. **공휴일 전후 피처**: 식수인원이 급감하는 연휴 전날/후날을 식별하여 피처화
3. **앙상블 모델링**: XGBoost와 CatBoost를 결합하여 모델 안정성 및 성능 향상
4. **결과 복원**: 참여율(Ratio) 예측 후 실제 인원수 환산 전략 유지

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None
from sklearn.metrics import mean_absolute_error
from korean_font import set_korean_font
import warnings

set_korean_font()
warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

한글 폰트 설정: Malgun Gothic (c:/Windows/Fonts/malgun.ttf)


In [2]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/sample_submission.csv')
weather_train = pd.read_csv('data/weather.csv')
weather_test = pd.read_csv('data/weatherTest.csv')

# 2. 날씨 데이터 통합 (필수 피처만 선택)
weather = pd.concat([weather_train, weather_test], axis=0)
weather['일시'] = pd.to_datetime(weather['일시'])
weather['강수량'] = weather['강수량'].fillna(0)
weather['비온날'] = (weather['강수량'] > 0).astype(int)
weather = weather[['일시', '기온', '비온날']]

# 3. 데이터 결합
train['일자'] = pd.to_datetime(train['일자'])
test['일자'] = pd.to_datetime(test['일자'])
train = pd.merge(train, weather, left_on='일자', right_on='일시', how='left')
test = pd.merge(test, weather, left_on='일자', right_on='일시', how='left')
train.drop(['일시'], axis=1, inplace=True)
test.drop(['일시'], axis=1, inplace=True)

print("데이터 결합 완료:", train.shape, test.shape)

데이터 결합 완료: (1220, 14) (50, 12)


In [3]:
# 4. 피처 엔지니어링: 연휴 전후 및 식사가능자
def feature_engineering(df):
    df['월'] = df['일자'].dt.month
    df['요일'] = df['일자'].dt.weekday
    df['일'] = df['일자'].dt.day
    
    # 식사가능자수
    df['식사가능자수'] = df['본사정원수'] - (df['본사휴가자수'] + df['본사출장자수'] + df['현본사소속재택근무자수'])
    
    # 공휴일 전후 피처
    df = df.sort_values('일자')
    df['일자간격'] = df['일자'].diff().dt.days
    df['공휴일후날'] = (df['일자간격'] > 2).astype(int)
    df['공휴일전날'] = (df['일자'].shift(-1).diff().dt.days > 2).astype(int)
    
    return df.fillna(0)

train = feature_engineering(train)
test = feature_engineering(test)
train['중식참여율'] = train['중식계'] / train['식사가능자수']
train['석식참여율'] = train['석식계'] / train['식사가능자수']

print("피처 엔지니어링 완료")

피처 엔지니어링 완료


In [4]:
# 5. 모델링: XGBoost + CatBoost
features = ['월', '요일', '식사가능자수', '본사출장자수', '본사시간외근무명령서승인건수', 
            '기온', '비온날', '공휴일전날', '공휴일후날']

x_train = train[features]
y_lunch = train['중식참여율']
y_dinner = train['석식참여율']
x_test = test[features]

xgb_params = {'n_estimators': 2000, 'learning_rate': 0.02, 'max_depth': 6, 'random_state': 42}
cb_params = {'n_estimators': 2000, 'learning_rate': 0.02, 'depth': 6, 'random_state': 42, 'verbose': 0}

def get_ensemble_pred(x_tr, y_tr, x_te):
    model_xgb = XGBRegressor(**xgb_params)
    model_xgb.fit(x_tr, y_tr)
    p_xgb = model_xgb.predict(x_te)
    
    if CatBoostRegressor:
        model_cb = CatBoostRegressor(**cb_params)
        model_cb.fit(x_tr, y_tr)
        p_cb = model_cb.predict(x_te)
        return (p_xgb + p_cb) / 2
    return p_xgb

print("예측 모델 학습 중...")
submission['중식계'] = get_ensemble_pred(x_train, y_lunch, x_test) * test['식사가능자수']
submission['석식계'] = get_ensemble_pred(x_train, y_dinner, x_test) * test['식사가능자수']

submission.to_csv('submission/submission_v10.csv', index=False)
print("제출 파일 저장 완료: submission/submission_v10.csv")

예측 모델 학습 중...
제출 파일 저장 완료: submission/submission_v10.csv
